### Install dependencies

```bash
git clone https://github.com/ChenDelong1999/subobjects.git
cd subobjects

conda create -n subobjects python=3.11 -y
conda activate subobjects
pip install -r requirements.txt
```

In [1]:
import random
import matplotlib.pyplot as plt

from subobjects.token_segmentation import get_segmenter
from subobjects.utils import visualize_masks

random.seed(42)
RESOLUTION=1024

In [ ]:
patch_segmenter = get_segmenter(
    type='patch', 
    patch_per_side=16,
    )

superpixel_segmenter = get_segmenter(
    type='superpixel', 
    max_tokens=100
    )

panoptic_segmenter = get_segmenter(
    type='panoptic', 
    checkpoint="facebook/mask2former-swin-tiny-coco-panoptic",
    )

epoc_segmenter = get_segmenter(
    type='epoc', 
    checkpoint="chendelong/DirectSAM-b0-1024px-sa1b-2ep-1017",
    threshold=0.1,
    )

In [3]:
from datasets import load_dataset

dataset = load_dataset("chendelong/HEIT", split='SA1B')
sample = dataset[random.randint(0, len(dataset) - 1)]
image = sample['image'].resize((RESOLUTION,RESOLUTION))

In [ ]:
plt.figure(figsize=(20, 5))
plt.subplot(1, 5, 1)
plt.imshow(image)
plt.axis('off')
plt.title('Input Image')

for i, segmenter in enumerate([patch_segmenter, superpixel_segmenter, panoptic_segmenter, epoc_segmenter]):
    masks = segmenter(image).cpu().numpy()[0]

    plt.subplot(1, 5, i+2)
    plt.imshow(visualize_masks(image, masks))
    plt.axis('off')
    plt.title(segmenter.__class__.__name__)

plt.show()